In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 20)

/Users/mironshoh/Desktop/workspaces/projects/personal/CineMatch/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
processed_dir = Path('../data/processed')
full_dir = Path('../data/full_dataset')
model_dir = Path('../models')
model_dir.mkdir(parents=True, exist_ok=True)

In [3]:
# Load already-filtered ratings (created by data_prep.ipynb)
ratings_f = pd.read_csv(processed_dir / 'ratings_filtered.csv')
print('ratings_f:', ratings_f.shape)

# Load movie metadata for inference
movies = pd.read_csv(full_dir / 'movies.csv')
print('movies:', movies.shape)

ratings_f: (31498689, 6)
movies: (87585, 3)


In [4]:
# Limit BLAS threads for implicit (avoids severe slowdowns)
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'

# Encode user and movie indices
user_ids = ratings_f['userId'].unique()
movie_ids = ratings_f['movieId'].unique()

user_id_to_idx = {uid: i for i, uid in enumerate(user_ids)}
movie_id_to_idx = {mid: i for i, mid in enumerate(movie_ids)}

ratings_f['user_idx'] = ratings_f['userId'].map(user_id_to_idx)
ratings_f['movie_idx'] = ratings_f['movieId'].map(movie_id_to_idx)

n_users = len(user_ids)
n_movies = len(movie_ids)
print(f'Users: {n_users}, Movies: {n_movies}')

Users: 200947, Movies: 16034


In [5]:
# Confidence-weighted implicit feedback
# preference = rating >= 3.5 (softer threshold)
# confidence = 1 + (rating - 3.5)  (higher rating = higher weight)
THRESHOLD = 3.5
ratings_f['confidence'] = np.where(
    ratings_f['rating'] >= THRESHOLD,
    1.0 + (ratings_f['rating'] - THRESHOLD),
    0.0
)
positive = ratings_f[ratings_f['confidence'] > 0]
print(f'Positive interactions (conf > 0): {len(positive)}, vs {len(ratings_f)} total')

Positive interactions (conf > 0): 19996990, vs 31498689 total


In [6]:
# Build sparse user-item matrix with confidence weights
X = csr_matrix(
    (positive['confidence'].values.astype(np.float32), (positive['user_idx'], positive['movie_idx'])),
    shape=(n_users, n_movies)
)

print('Matrix shape:', X.shape)
print('Non-zeros:', X.nnz)
print('Mean confidence of non-zeros: {:.2f}'.format(X.data.mean()))
print('Sparsity: {:.4f}%'.format(100 * X.nnz / (X.shape[0] * X.shape[1])))

Matrix shape: (200947, 16034)
Non-zeros: 19996990
Mean confidence of non-zeros: 1.70
Sparsity: 0.6206%


In [7]:
# Best config from hyperparameter grid search
BEST_FACTORS = 256
BEST_REG = 0.1
BEST_ALPHA = 20
ITERATIONS = 20

print(f'Training ALS: factors={BEST_FACTORS}, reg={BEST_REG}, alpha={BEST_ALPHA}, iterations={ITERATIONS}')

model = AlternatingLeastSquares(
    factors=BEST_FACTORS,
    regularization=BEST_REG,
    iterations=ITERATIONS,
    random_state=42
)

model.fit(X * BEST_ALPHA)

Training ALS: factors=256, reg=0.1, alpha=20, iterations=20


/Users/mironshoh/Desktop/workspaces/projects/personal/CineMatch/.venv/lib/python3.9/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 20/20 [14:56<00:00, 44.80s/it]


In [ ]:
# Build reverse mapping: movie_idx -> movieId
idx_to_movie_id = {v: k for k, v in movie_id_to_idx.items()}

# Build movie metadata lookup (movieId -> title, genres)
movie_meta = movies.set_index('movieId')[['title', 'genres']].to_dict('index')

model_artifacts = {
    'model': model,
    'user_id_to_idx': user_id_to_idx,
    'movie_id_to_idx': movie_id_to_idx,
    'idx_to_movie_id': idx_to_movie_id,
    'movie_meta': movie_meta,
    'config': {
        'factors': BEST_FACTORS,
        'regularization': BEST_REG,
        'alpha': BEST_ALPHA,
        'iterations': ITERATIONS,
    }
}

model_path = model_dir / 'als_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model_artifacts, f)

print(f'Model + artifacts saved to {model_path}')
print(f'Model size: {model_path.stat().st_size / 1024 / 1024:.1f} MB')

Model + artifacts saved to ../models/als_model_v2.pkl
Model size: 221.2 MB


In [9]:
# Quick sanity check: recommend for a sample user
sample_user_id = 1  # userId 1
if sample_user_id in user_id_to_idx:
    user_idx = user_id_to_idx[sample_user_id]
    recs = model.recommend(user_idx, X[user_idx], N=5, filter_already_liked_items=True)
    rec_ids = [int(r) if isinstance(r, (np.integer, int)) else int(r[0]) for r in recs[0]]
    rec_movie_ids = [idx_to_movie_id[idx] for idx in rec_ids]
    print(f'Top 5 recommendations for user {sample_user_id}:')
    for mid in rec_movie_ids:
        meta = movie_meta.get(mid, {})
        print(f'  {meta.get("title", "?")}  ({meta.get("genres", "?")})')
else:
    print(f'User {sample_user_id} not in filtered dataset')

Top 5 recommendations for user 1:
  Annie Hall (1977)  (Comedy|Romance)
  Bridge on the River Kwai, The (1957)  (Adventure|Drama|War)
  American Beauty (1999)  (Drama|Romance)
  Raising Arizona (1987)  (Comedy)
  Rain Man (1988)  (Drama)


In [ ]:
# Create leave-one-out test set using rating >= 4.0 as relevance
if not pd.api.types.is_datetime64_any_dtype(ratings_f['timestamp']):
    ratings_f['timestamp'] = pd.to_datetime(ratings_f['timestamp'], unit='s')

relevance = ratings_f[ratings_f['rating'] >= 4.0].sort_values(['user_idx', 'timestamp'])
test_holdout = relevance.groupby('user_idx').tail(1)
train_likes = relevance.drop(test_holdout.index)

# Sample 5000 users for evaluation (speeds up comparison significantly)
rng = np.random.default_rng(42)
eval_users = rng.choice(test_holdout['user_idx'].unique(),
                       size=min(5000, len(test_holdout)), replace=False)
test_sample = test_holdout[test_holdout['user_idx'].isin(eval_users)]

print(f'Test items (leave-one-out): {len(test_holdout)}, sampling {len(test_sample)} for eval')

In [ ]:
# Build binary training matrix for evaluation
X_eval = csr_matrix(
    (np.ones(len(train_likes), dtype=np.float32),
     (train_likes['user_idx'], train_likes['movie_idx'])),
    shape=(n_users, n_movies)
)
print(f'Eval matrix: {X_eval.shape}, non-zeros: {X_eval.nnz}')

In [ ]:
def evaluate_model(model, X_mat, test_df, k=10):
    users = test_df['user_idx'].unique()
    test_map = test_df.set_index('user_idx')['movie_idx'].to_dict()

    precisions, recalls, aps = [], [], []
    for user_idx in users:
        relevant = {test_map[user_idx]}

        recs = model.recommend(user_idx, X_mat[user_idx], N=k,
                               filter_already_liked_items=True)
        if isinstance(recs, tuple):
            recs_items = [int(r) for r in recs[0]]
        else:
            recs_items = [int(r[0]) for r in recs]

        hits = [1 if r in relevant else 0 for r in recs_items]
        precisions.append(sum(hits) / k)
        recalls.append(sum(hits) / len(relevant))

        cum = 0
        ap = 0.0
        for i, h in enumerate(hits, 1):
            if h:
                cum += 1
                ap += cum / i
        aps.append(ap / min(len(relevant), k))

    return {
        'precision@10': float(np.mean(precisions)),
        'recall@10': float(np.mean(recalls)),
        'map@10': float(np.mean(aps)),
        'users_evaluated': len(precisions)
    }

In [ ]:
# Load both models
v1_path = model_dir / 'als_model.pkl'
v2_path = model_dir / 'als_model_v2.pkl'

models = {}
for label, path in [('v1 (binary >=4.0)', v1_path), ('v2 (conf-weighted >=3.5)', v2_path)]:
    with open(path, 'rb') as f:
        models[label] = pickle.load(f)['model']
    print(f'Loaded {label}')

print(f'\nEvaluating {len(test_sample)} users... (this takes a while)')

In [ ]:
results = {}
for label, model in models.items():
    results[label] = evaluate_model(model, X_eval, test_sample, k=10)
    r = results[label]
    print(f'{label:>30s}  prec@10={r["precision@10"]:.4f}  recall@10={r["recall@10"]:.4f}  '
          f'map@10={r["map@10"]:.4f}  users={r["users_evaluated"]}')

In [ ]:
cmp = pd.DataFrame(results).T
cmp['precision@10'] = cmp['precision@10'].map('{:.4f}'.format)
cmp['recall@10'] = cmp['recall@10'].map('{:.4f}'.format)
cmp['map@10'] = cmp['map@10'].map('{:.4f}'.format)
print(cmp[['precision@10', 'recall@10', 'map@10', 'users_evaluated']])

if results:
    best = max(results, key=lambda k: results[k]['map@10'])
    print(f'\nWinner by MAP@10: {best}')